In [ ]:
!rm -rf "ff_frames"

In [ ]:
import os, urllib, urllib.request, tempfile, time, sys, json, random
from tqdm import tqdm
from os.path import join
import unittest.mock as mock

In [ ]:


# # ── Constants (from original script) ──────────────────────────────────────
# FILELIST_URL = 'misc/filelist.json'
# DEEPFEAKES_DETECTION_URL = 'misc/deepfake_detection_filenames.json'
# DEEPFAKES_MODEL_NAMES = ['decoder_A.h5', 'decoder_B.h5', 'encoder.h5']

# DATASETS = {
#     'original': 'original_sequences/youtube',
#     'Deepfakes': 'manipulated_sequences/Deepfakes',
#     'Face2Face': 'manipulated_sequences/Face2Face',
#     'FaceSwap':  'manipulated_sequences/FaceSwap',
#     'NeuralTextures': 'manipulated_sequences/NeuralTextures',
#     'DeepFakeDetection_original': 'original_sequences/actors',
#     'DeepFakeDetection': 'manipulated_sequences/DeepFakeDetection',
#     'FaceShifter': 'manipulated_sequences/FaceShifter',
# }
# ALL_DATASETS = ['original', 'Deepfakes', 'Face2Face', 'FaceSwap',
#                 'NeuralTextures', 'DeepFakeDetection_original',
#                 'DeepFakeDetection', 'FaceShifter']

# # ── Helper functions ───────────────────────────────────────────────────────
# def reporthook(count, block_size, total_size):
#     global start_time
#     if count == 0:
#         start_time = time.time()
#         return
#     duration = time.time() - start_time + 1e-8
#     progress_size = int(count * block_size)
#     speed = int(progress_size / (1024 * duration))
#     percent = min(int(count * block_size * 100 / total_size), 100)
#     sys.stdout.write(f"\r  {percent}%  {progress_size/1024/1024:.1f} MB  {speed} KB/s  {duration:.0f}s")
#     sys.stdout.flush()

# def download_file(url, out_file, report_progress=False):
#     os.makedirs(os.path.dirname(out_file), exist_ok=True)
#     if not os.path.isfile(out_file):
#         fh, tmp = tempfile.mkstemp(dir=os.path.dirname(out_file))
#         os.close(fh)
#         if report_progress:
#             urllib.request.urlretrieve(url, tmp, reporthook=reporthook)
#         else:
#             urllib.request.urlretrieve(url, tmp)
#         os.rename(tmp, out_file)
#     else:
#         print(f'  Skipping (exists): {out_file}')

# def download_files(filenames, base_url, output_path):
#     os.makedirs(output_path, exist_ok=True)
#     for filename in tqdm(filenames):
#         download_file(base_url + filename, join(output_path, filename))

# # ── YOUR SETTINGS — edit these ─────────────────────────────────────────────
# OUTPUT_PATH  = '/content/drive/MyDrive/data'
# DATASETS_TO_DL = ['original', 'Deepfakes', 'Face2Face', 'FaceSwap']  # subset
# COMPRESSION  = 'c23'       # c23 = smaller files, good quality
# TYPE         = 'videos'
# NUM_VIDEOS   = 125          # ← keep small to test; remove limit for full download
# BASE_URL     = 'http://kaldir.vc.in.tum.de/faceforensics/v3/'

# os.makedirs(OUTPUT_PATH, exist_ok=True)

# # ── Download loop ──────────────────────────────────────────────────────────
# for dataset in DATASETS_TO_DL:
#     dataset_path = DATASETS[dataset]
#     print(f'\n{"="*55}')
#     print(f'  Downloading: {dataset}  [{COMPRESSION}/{TYPE}]')
#     print(f'{"="*55}')

#     # Get file list from server
#     if 'original' in dataset_path and 'actors' not in dataset_path:
#         file_pairs = json.loads(
#             urllib.request.urlopen(BASE_URL + FILELIST_URL).read().decode()
#         )
#         filelist = []
#         for pair in file_pairs:
#             filelist += pair
#     elif 'actors' in dataset_path:
#         filepaths = json.loads(
#             urllib.request.urlopen(BASE_URL + DEEPFEAKES_DETECTION_URL).read().decode()
#         )
#         filelist = filepaths['actors']
#     else:
#         file_pairs = json.loads(
#             urllib.request.urlopen(BASE_URL + FILELIST_URL).read().decode()
#         )
#         filelist = []
#         for pair in file_pairs:
#             filelist.append('_'.join(pair))
#             filelist.append('_'.join(pair[::-1]))

#     # Limit number of videos
#     if NUM_VIDEOS:
#         filelist = filelist[:NUM_VIDEOS]
#         print(f'  Limiting to first {NUM_VIDEOS} videos')

#     filelist = [f + '.mp4' for f in filelist]

#     video_url = BASE_URL + f'{dataset_path}/{COMPRESSION}/videos/'
#     out_path  = join(OUTPUT_PATH, dataset_path, COMPRESSION, 'videos')

#     print(f'  URL    : {video_url}')
#     print(f'  Saving : {out_path}')
#     download_files(filelist, video_url, out_path)
#     print(f'  ✅ Done: {dataset}')

# print('\n\n✅ ALL DOWNLOADS COMPLETE')
# print(f'Files saved to: {OUTPUT_PATH}')

# # ── Verify what was downloaded ─────────────────────────────────────────────
# from pathlib import Path
# print('\n=== DOWNLOAD SUMMARY ===')
# for dataset in DATASETS_TO_DL:
#     p = Path(OUTPUT_PATH) / DATASETS[dataset] / COMPRESSION / 'videos'
#     if p.exists():
#         files = list(p.glob('*.mp4'))
#         print(f'  {dataset:<30} : {len(files)} mp4 files')
#     else:
#         print(f'  {dataset:<30} : ❌ folder not found')

# Deepfake Detection Pipeline
### Vision Transformer + Stationary Wavelet Transform on FaceForensics++

**Pipeline Overview:**
1. Frame Extraction (ffmpeg @ 0.2 fps)
2. Face Detection + Alignment (MTCNN)
3. Face Segmentation (BiSeNet)
4. Stationary Wavelet Transform (SWT)
5. ViT Classification (4 classes: real / deepfake / faceswap / face2face)

> **FaceForensics++ requires registration** at https://github.com/ondyari/FaceForensics. After downloading, upload/mount your dataset and update `FF_ROOT` below.

## Cell 1 - Install Dependencies

In [ ]:
from google.colab import drive
import os

if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive already mounted!")
else:
    drive.mount('/content/drive')

In [ ]:

!pip install numpy>=2.0.0 --upgrade # Ensure numpy is updated to a compatible version

In [ ]:
# Install all required packages
!pip uninstall Pillow -y # Force uninstall any existing Pillow version
!pip install -q \
    numpy==1.26.4 \
    Pillow==10.2.0 \
    scipy==1.11.4 \
    scikit-learn==1.3.2 \
    facenet-pytorch \
    timm \
    PyWavelets \
    opencv-python-headless \
    albumentations \
    tqdm \
    gdown

# ffmpeg is pre-installed on Colab; verify
!ffmpeg -version 2>&1 | head -1
print('All dependencies installed.')

## Cell 2 - Imports and Global Configuration

In [ ]:
# Standard library
import os
import json
import math
import random
import shutil
import warnings
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')

# Numerical / image
import numpy as np
import cv2
import pywt                      # Stationary Wavelet Transform
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

# Face detection
from facenet_pytorch import MTCNN

# ViT backbone
import timm

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report
)

# Utilities
from tqdm import tqdm

# ─── GLOBAL CONFIGURATION ──────────────────────────────────────────────────
CFG = {
    # Dataset root (FaceForensics++ structure expected)
    'FF_ROOT': '',

    # Frame and face settings
    'FRAME_DIR': '/content/ff_frames',       # extracted frame cache
    'FPS': 1,                               # 1 frame per 5 seconds  (paper: 1/5 fps)
    'FACE_SIZE': 224,                         # ViT input resolution
    'MAX_FRAMES_PER_VIDEO': 50,              # ≥100 per paper recommendation

    # Model  ← CHANGED: vit_base → vit_small (88 M → 22 M params; more Colab-stable)
    'MODEL_NAME': 'vit_small_patch16_224',
    'NUM_CLASSES': 4,                         # real / deepfake / faceswap / face2face
    'LABEL_MAP': {0: 'real', 1: 'deepfake', 2: 'faceswap', 3: 'face2face'},

    # Training
    'BATCH_SIZE': 32,          # ←
    'LR': 1e-4,
    'EPOCHS': 25,
    'PATIENCE': 8,             # early-stopping patience
    'SEED': 42,

    # Split ratios
    'TRAIN_RATIO': 0.70,
    'VAL_RATIO':   0.15,
    # TEST_RATIO = 1 − TRAIN_RATIO − VAL_RATIO = 0.15

    # Wavelet  (paper: single-level Haar SWT)
    'WAVELET': 'haar',
    'SWT_LEVEL': 1,

    # Checkpointing
    'CHECKPOINT_DIR': '/content/checkpoints',
}

# ─── Reproducibility ───────────────────────────────────────────────────────
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['SEED'])

# Device — automatically use GPU if available
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

# Create output directories
Path(CFG['FRAME_DIR']).mkdir(parents=True, exist_ok=True)
Path(CFG['CHECKPOINT_DIR']).mkdir(parents=True, exist_ok=True)
print('Config ready.')

In [ ]:
import shutil
# Clear old frame cache so new FPS takes effect
shutil.rmtree('/content/ff_frames', ignore_errors=True)
shutil.rmtree('/content/ff_faces', ignore_errors=True)
print("✅ Cache cleared — ready for re-extraction!")

In [ ]:
# ── CHECKPOINT CONFIG — saves everything to Drive automatically ────────────
DRIVE_CKPT_DIR = '/content/drive/MyDrive/data/checkpoints'
import os
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

# Override checkpoint paths to point to Drive directly
CFG['CHECKPOINT_DIR'] = DRIVE_CKPT_DIR
BEST_CKPT = os.path.join(DRIVE_CKPT_DIR, 'best_model.pth')
LAST_CKPT = os.path.join(DRIVE_CKPT_DIR, 'last_model.pth')

print(f'✅ Checkpoints will save directly to Drive:')
print(f'   {DRIVE_CKPT_DIR}')

## Cell 3 - Dataset Discovery and Video-Level Split

We collect video paths per class and perform a **strict video-level split** so that
no frames from the same video appear in both train and val/test sets.

In [ ]:
# FaceForensics++ folder structure mapping
from typing import Dict, List, Tuple
#added by SHIVVVVVVVVVVVVV
FF_SOURCES = {
    0: '/content/drive/MyDrive/data/original_sequences/youtube/c23/videos',
    1: '/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos',
    2: '/content/drive/MyDrive/data/manipulated_sequences/FaceSwap/c23/videos',
    3: '/content/drive/MyDrive/data/manipulated_sequences/Face2Face/c23/videos',
}


def discover_videos(ff_root: str, sources: Dict[int, str]) -> Dict[int, List[str]]:
    """
    Walk the FaceForensics++ directory tree and collect video paths per label.
    Returns {label_id: [video_path, ...]}
    """
    videos_by_label: Dict[int, List[str]] = defaultdict(list)
    for label, rel_path in sources.items():
        folder = Path(ff_root) / rel_path
        if not folder.exists():
            print(f'WARNING: Folder not found (label={label}): {folder}')
            continue
        found = sorted(folder.glob('*.mp4')) + sorted(folder.glob('*.avi'))
        videos_by_label[label] = [str(p) for p in found]
        print(f'  Label {label} ({CFG["LABEL_MAP"][label]:>10}): {len(found)} videos')
    return dict(videos_by_label)


def video_level_split(
    videos_by_label: Dict[int, List[str]],
    train_ratio: float = 0.70,
    val_ratio:   float = 0.15,
    seed: int = 42,
) -> Tuple[List[Tuple], List[Tuple], List[Tuple]]:
    """
    Perform a deterministic, stratified, VIDEO-LEVEL split.

    Stratified: same train/val/test proportion within each class.
    Frame-level leakage is impossible because the split operates on video IDs,
    NOT on individual frames -- all frames of a video stay in one split.

    Returns three lists of (video_path, label) tuples.
    """
    rng = random.Random(seed)
    train, val, test = [], [], []

    for label, paths in videos_by_label.items():
        shuffled = paths.copy()
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_train = math.floor(n * train_ratio)
        n_val   = math.floor(n * val_ratio)
        # remainder goes to test
        train += [(p, label) for p in shuffled[:n_train]]
        val   += [(p, label) for p in shuffled[n_train:n_train + n_val]]
        test  += [(p, label) for p in shuffled[n_train + n_val:]]

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test


print('Discovering videos ...')
VIDEOS_BY_LABEL = discover_videos(CFG['FF_ROOT'], FF_SOURCES)

TRAIN_VIDS, VAL_VIDS, TEST_VIDS = video_level_split(
    VIDEOS_BY_LABEL,
    train_ratio=CFG['TRAIN_RATIO'],
    val_ratio=CFG['VAL_RATIO'],
    seed=CFG['SEED'],
)

print(f'\nSplit summary (video-level, no frame leakage):')
print(f'  Train : {len(TRAIN_VIDS):>4} videos')
print(f'  Val   : {len(VAL_VIDS):>4} videos')
print(f'  Test  : {len(TEST_VIDS):>4} videos')

## Cell 4 - Frame Extraction (ffmpeg @ 0.2 fps)

In [ ]:
def extract_frames(
    video_path: str,
    out_dir: str,
    fps: float = 0.2,
    max_frames: Optional[int] = None,
) -> List[str]:
    """
    Use ffmpeg to extract frames from a video at the specified fps rate.

    fps=0.2 means 1 frame every 5 seconds -- enough temporal coverage
    while keeping disk usage and compute manageable.

    Edge cases handled:
    - Corrupted videos: ffmpeg returns non-zero exit code -> return []
    - max_frames cap: evenly sub-samples if more frames are extracted

    Args:
        video_path : path to source video file
        out_dir    : directory to write PNG frames into
        fps        : extraction frame rate (0.2 = 1 frame per 5 s)
        max_frames : optional cap on total frames kept
    Returns:
        Sorted list of extracted frame file paths.
    """
    os.makedirs(out_dir, exist_ok=True)
    pattern = os.path.join(out_dir, 'frame_%04d.png')

    cmd = [
        'ffmpeg', '-y',
        '-i', video_path,
        '-vf', f'fps={fps}',
        '-q:v', '2',          # near-lossless quality
        pattern,
    ]
    result = subprocess.run(
        cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    if result.returncode != 0:
        # Corrupted or unreadable video -- skip gracefully
        return []

    frames = sorted(Path(out_dir).glob('frame_*.png'))
    frames = [str(f) for f in frames]

    if max_frames and len(frames) > max_frames:
        # Evenly sub-sample to max_frames to avoid spending too much time per video
        idxs = np.linspace(0, len(frames) - 1, max_frames, dtype=int)
        frames = [frames[i] for i in idxs]

    return frames


def extract_all_frames(
    video_label_list: List[Tuple[str, int]],
    base_dir: str,
    fps: float = 0.2,
    max_frames: Optional[int] = None,
) -> List[Tuple[str, int]]:
    """
    Extract frames for a list of (video_path, label) tuples.

    Caching: if the output folder already exists and is non-empty,
    extraction is skipped and the existing frames are returned.
    This makes re-runs on Colab much faster.

    Returns:
        List of (frame_path, label) pairs for all extracted frames.
    """
    frame_label_pairs: List[Tuple[str, int]] = []

    for video_path, label in tqdm(video_label_list, desc='Extracting frames'):
        video_stem = Path(video_path).stem
        out_dir = os.path.join(base_dir, f'label_{label}', video_stem)

        # Cache check: skip extraction if folder already populated
        existing = sorted(Path(out_dir).glob('frame_*.png')) if Path(out_dir).exists() else []
        if existing:
            frames = [str(f) for f in existing]
        else:
            frames = extract_frames(video_path, out_dir, fps, max_frames)

        frame_label_pairs.extend([(fp, label) for fp in frames])

    return frame_label_pairs


print('Extracting frames from TRAIN videos ...')
TRAIN_FRAMES = extract_all_frames(TRAIN_VIDS, CFG['FRAME_DIR'], CFG['FPS'], CFG['MAX_FRAMES_PER_VIDEO'])

print('Extracting frames from VAL videos ...')
VAL_FRAMES = extract_all_frames(VAL_VIDS, CFG['FRAME_DIR'], CFG['FPS'], CFG['MAX_FRAMES_PER_VIDEO'])

print('Extracting frames from TEST videos ...')
TEST_FRAMES = extract_all_frames(TEST_VIDS, CFG['FRAME_DIR'], CFG['FPS'], CFG['MAX_FRAMES_PER_VIDEO'])

print(f'\nFrame counts:')
print(f'  Train : {len(TRAIN_FRAMES):>6} frames')
print(f'  Val   : {len(VAL_FRAMES):>6} frames')
print(f'  Test  : {len(TEST_FRAMES):>6} frames')

## Cell 5 - Face Detection, Alignment (MTCNN)

In [ ]:
MTCNN_DETECTOR = MTCNN(
    image_size=224,
    margin=40,
    min_face_size=30,
    thresholds=[0.5, 0.6, 0.6],
    factor=0.709,
    keep_all=False,
    device=DEVICE,
    select_largest=True,
    post_process=False,
)

face_detection_stats = {'total': 0, 'detected': 0}


def align_face(img, landmarks, target_size=224):
    """Align face crop so the eye line is horizontal."""
    left_eye, right_eye = landmarks[0], landmarks[1]
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    angle = math.degrees(math.atan2(dy, dx))
    eye_center = ((left_eye[0] + right_eye[0]) / 2,
                  (left_eye[1] + right_eye[1]) / 2)
    M = cv2.getRotationMatrix2D(eye_center, angle, scale=1.0)
    aligned = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]),
                              flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REFLECT_101)
    return cv2.resize(aligned, (target_size, target_size))


def detect_and_align(frame_path, detector, target_size=224):
    """
    Detect + align the primary face in a frame.
    Returns HxWx3 uint8 RGB numpy array, or None if no face found.
    Called ONLY in the main process — never inside a DataLoader worker.
    """
    face_detection_stats['total'] += 1

    img_bgr = cv2.imread(frame_path)
    if img_bgr is None:
        return None

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)

    try:
        boxes, probs, landmarks = detector.detect(pil_img, landmarks=True)
    except Exception:
        return None

    if boxes is None or len(boxes) == 0:
        return None   # strict skip — no fallback crops

    best_idx = int(np.argmax(probs))
    box = boxes[best_idx].astype(int)
    lm  = landmarks[best_idx]

    h, w = img_rgb.shape[:2]
    x1 = max(0, box[0]); y1 = max(0, box[1])
    x2 = min(w, box[2]); y2 = min(h, box[3])
    face_crop = img_rgb[y1:y2, x1:x2]

    if face_crop.size == 0:
        return None

    face_resized = cv2.resize(face_crop, (target_size, target_size))
    lm_crop = lm - np.array([x1, y1])
    sx = target_size / max(x2 - x1, 1)
    sy = target_size / max(y2 - y1, 1)
    lm_scaled = lm_crop * np.array([sx, sy])

    aligned = align_face(face_resized, lm_scaled, target_size)
    face_detection_stats['detected'] += 1
    return aligned


def pre_extract_faces(
    frame_label_pairs: List[Tuple[str, int]],
    face_dir: str,
    detector: MTCNN,
    target_size: int = 224,
) -> List[Tuple[str, int]]:
    """
    Run MTCNN on every frame IN THE MAIN PROCESS and save valid face crops
    to disk as PNGs.

    Returns a list of (face_png_path, label) for every frame where a face
    was successfully detected.  Frames without a face are simply omitted.

    This is the key fix:
      • CUDA-based MTCNN runs safely in the main process.
      • DataLoader workers only do disk-read + SWT — zero CUDA calls.
      • Face crops are cached, so re-running the notebook is fast.
    """
    os.makedirs(face_dir, exist_ok=True)
    face_pairs: List[Tuple[str, int]] = []

    for frame_path, label in tqdm(frame_label_pairs, desc=f'Detecting faces → {face_dir}'):
        # Deterministic save path derived from the frame path
        rel  = Path(frame_path).relative_to(CFG['FRAME_DIR'])  # label_X/video/frame_NNNN.png
        dest = Path(face_dir) / rel
        dest.parent.mkdir(parents=True, exist_ok=True)

        if dest.exists():
            # Already extracted on a previous run — skip MTCNN
            face_pairs.append((str(dest), label))
            continue

        face = detect_and_align(frame_path, detector, target_size)
        if face is None:
            continue  # no face found → skip this frame entirely

        # Save as PNG (lossless)
        cv2.imwrite(str(dest), cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
        face_pairs.append((str(dest), label))

    total    = face_detection_stats['total']
    detected = face_detection_stats['detected']
    print(f'\n[Face Detection Stats]')
    print(f'  Frames processed : {total}')
    print(f'  Faces detected   : {detected}')
    print(f'  Detection rate   : {detected / max(total, 1):.2%}')
    print(f'  Valid face pairs : {len(face_pairs)}')
    return face_pairs


# ── Run face pre-extraction for all three splits ───────────────────────────
FACE_DIR = '/content/ff_faces'   # separate cache dir for face crops

print('Pre-extracting TRAIN faces (main process, CUDA OK) ...')
face_detection_stats['total'] = face_detection_stats['detected'] = 0
TRAIN_FACES = pre_extract_faces(TRAIN_FRAMES, FACE_DIR, MTCNN_DETECTOR, CFG['FACE_SIZE'])

print('\nPre-extracting VAL faces ...')
face_detection_stats['total'] = face_detection_stats['detected'] = 0
VAL_FACES = pre_extract_faces(VAL_FRAMES, FACE_DIR, MTCNN_DETECTOR, CFG['FACE_SIZE'])

print('\nPre-extracting TEST faces ...')
face_detection_stats['total'] = face_detection_stats['detected'] = 0
TEST_FACES = pre_extract_faces(TEST_FRAMES, FACE_DIR, MTCNN_DETECTOR, CFG['FACE_SIZE'])

print(f'\n[Pre-extraction complete]')
print(f'  Train face samples : {len(TRAIN_FACES)}')
print(f'  Val   face samples : {len(VAL_FACES)}')
print(f'  Test  face samples : {len(TEST_FACES)}')

if len(TRAIN_FACES) == 0:
    raise RuntimeError(
        'TRAIN_FACES is empty — MTCNN found zero faces.\n'
        'Check that FRAME_DIR contains valid PNG frames and that\n'
        'the videos contain clearly visible faces.'
    )



## Cell 6 - Face Segmentation (BiSeNet)

BiSeNet (Bilateral Segmentation Network) performs face parsing to isolate
**skin, eyes, eyebrows, nose, and mouth** while zeroing out background, hair,
and accessories. This focuses the wavelet analysis purely on facial tissue.

In [ ]:
# @title
# # ---------------------------------------------------------------------------
# # BiSeNet-V1 face parsing (CelebAMask-HQ 19-class)
# # Architecture reference: arxiv.org/abs/1808.00897
# # ---------------------------------------------------------------------------

# class ConvBNReLU(nn.Module):
#     """Conv2d -> BatchNorm -> ReLU building block."""
#     def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
#         super().__init__()
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_ch, out_ch, kernel, stride, padding, bias=False),
#             nn.BatchNorm2d(out_ch),
#             nn.ReLU(inplace=True),
#         )
#     def forward(self, x):
#         return self.conv(x)


# class SpatialPath(nn.Module):
#     """Captures fine-grained spatial detail via consecutive strided convolutions."""
#     def __init__(self):
#         super().__init__()
#         self.layers = nn.Sequential(
#             ConvBNReLU(3,  64, stride=2),
#             ConvBNReLU(64, 64, stride=2),
#             ConvBNReLU(64, 64, stride=2),
#             ConvBNReLU(64, 128),
#         )
#     def forward(self, x):
#         return self.layers(x)


# class AttentionRefinementModule(nn.Module):
#     """ARM: refines context features using channel attention."""
#     def __init__(self, in_ch, out_ch):
#         super().__init__()
#         self.conv = ConvBNReLU(in_ch, out_ch)
#         self.attn = nn.Sequential(
#             nn.AdaptiveAvgPool2d(1),
#             nn.Conv2d(out_ch, out_ch, 1),
#             nn.BatchNorm2d(out_ch),
#             nn.Sigmoid(),
#         )
#     def forward(self, x):
#         feat = self.conv(x)
#         return feat * self.attn(feat)


# class FeatureFusionModule(nn.Module):
#     """FFM: fuses spatial-path and context-path features."""
#     def __init__(self, in_ch, out_ch):
#         super().__init__()
#         self.conv = ConvBNReLU(in_ch, out_ch)
#         self.attn = nn.Sequential(
#             nn.AdaptiveAvgPool2d(1),
#             nn.Conv2d(out_ch, out_ch // 4, 1),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(out_ch // 4, out_ch, 1),
#             nn.Sigmoid(),
#         )
#     def forward(self, sp, cp):
#         x = torch.cat([sp, cp], dim=1)
#         feat = self.conv(x)
#         return feat + feat * self.attn(feat)


# class BiSeNet(nn.Module):
#     """
#     Lightweight BiSeNet for face parsing with 19 semantic classes
#     (CelebAMask-HQ label set). Context path uses a ResNet-18 backbone.
#     """
#     def __init__(self, num_classes: int = 19):
#         super().__init__()
#         import torchvision.models as tvm
#         backbone = tvm.resnet18(weights='DEFAULT')

#         # Context path encoder
#         self.cp_layer0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
#         self.cp_layer1 = backbone.layer1   # 64  channels
#         self.cp_layer2 = backbone.layer2   # 128 channels
#         self.cp_layer3 = backbone.layer3   # 256 channels
#         self.cp_layer4 = backbone.layer4   # 512 channels

#         # Attention refinement on the two deepest context features
#         self.arm3 = AttentionRefinementModule(256, 128)
#         self.arm4 = AttentionRefinementModule(512, 128)
#         self.gap   = nn.AdaptiveAvgPool2d(1)   # global average pooling for context guidance

#         self.sp     = SpatialPath()                  # spatial path: 128 ch
#         self.ffm    = FeatureFusionModule(256, 256)  # fuse 128+128 -> 256
#         self.head   = nn.Conv2d(256, num_classes, 1) # pixel-wise classification

#     def forward(self, x):
#         H, W = x.shape[2], x.shape[3]

#         # Spatial path (preserves fine edge details)
#         sp = self.sp(x)

#         # Context path (captures global semantic context)
#         c = self.cp_layer0(x)
#         c = self.cp_layer1(c)
#         c = self.cp_layer2(c)
#         c3 = self.cp_layer3(c)
#         c4 = self.cp_layer4(c3)

#         gap = self.gap(c4)   # global scene context
#         a3 = F.interpolate(self.arm3(c3), size=sp.shape[2:], mode='bilinear', align_corners=True)
#         a4 = F.interpolate(self.arm4(c4) + gap, size=sp.shape[2:], mode='bilinear', align_corners=True)
#         cp = a3 + a4

#         # Feature fusion + upsample to input resolution
#         out = self.ffm(sp, cp)
#         out = F.interpolate(self.head(out), size=(H, W), mode='bilinear', align_corners=True)
#         return out


# # CelebAMask-HQ class indices to keep (face-relevant regions only)
# # 0=background  1=skin  2=left_brow  3=right_brow  4=left_eye  5=right_eye
# # 6=eyeglasses  7=left_ear  8=right_ear  10=nose  11=mouth
# # 12=upper_lip  13=lower_lip  17=hair  (hair EXCLUDED)
# FACE_KEEP_CLASSES = {1, 2, 3, 4, 5, 10, 11, 12, 13}   # skin + eyes + nose + mouth


# def build_segmenter(device: torch.device) -> BiSeNet:
#     """
#     Instantiate BiSeNet with pretrained ResNet-18 backbone weights.
#     Fine-tuned CelebAMask weights can be loaded via load_state_dict if available.
#     """
#     model = BiSeNet(num_classes=19).to(device)
#     model.eval()
#     return model


# BISENET = build_segmenter(DEVICE)

# # Optional: load fine-tuned CelebAMask-HQ weights if you have them
# # BISENET.load_state_dict(torch.load('/content/bisenet_celebamaskhq.pth', map_location=DEVICE))

# # ImageNet normalization constants for segmenter input
# _SEG_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
# _SEG_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)


# @torch.no_grad()
# def segment_face(
#     face_rgb: np.ndarray,
#     model: BiSeNet,
#     keep_classes: set = FACE_KEEP_CLASSES,
# ) -> np.ndarray:
#     """
#     Run BiSeNet face parsing on a 224x224 RGB image.
#     Background, hair, and accessories are zeroed out.

#     Returns:
#         Masked face as HxWx3 uint8 array (non-face regions = 0).
#     """
#     # Normalize to ImageNet stats
#     t = torch.from_numpy(face_rgb).permute(2, 0, 1).float().unsqueeze(0) / 255.0
#     t = t.to(DEVICE)
#     t = (t - _SEG_MEAN) / _SEG_STD

#     logits  = model(t)                                                    # (1, 19, H, W)
#     seg_map = logits.argmax(dim=1).squeeze(0).cpu().numpy()               # (H, W)

#     # Build binary mask: 1 for kept classes, 0 otherwise
#     mask = np.zeros_like(seg_map, dtype=np.uint8)
#     for cls in keep_classes:
#         mask[seg_map == cls] = 1

#     # Zero-out non-face regions
#     masked = face_rgb * mask[:, :, np.newaxis]
#     return masked.astype(np.uint8)


# print('BiSeNet segmenter ready.')

## Cell 7 - Stationary Wavelet Transform (SWT)

SWT (undecimated DWT) preserves spatial dimensions at each decomposition level,
making it ideal for artifact detection. Deepfake generation artifacts (blending
seams, GAN checkerboard patterns) produce distinctive high-frequency signatures
in the horizontal (LH), vertical (HL), and diagonal (HH) wavelet subbands.

In [ ]:

def apply_swt(
    face_rgb: np.ndarray,
    wavelet: str = 'haar',
    level: int = 1,
) -> np.ndarray:
    """
    Apply Stationary Wavelet Transform to a face image following the SWTVIT paper.

    Paper equation (§3.1):
        C = sqrt(H² + V² + D²)
    where H, V, D are the horizontal, vertical, and diagonal detail sub-bands
    from a single-level 2-D SWT decomposition.

    The approximation (low-frequency) sub-band is intentionally discarded; only
    the three high-frequency detail bands capture the subtle blending artefacts
    introduced by autoencoder-based deepfake generation.

    The combined matrix C is then replicated across 3 channels so the ViT
    backbone (expecting RGB input) receives a valid (224, 224, 3) tensor.

    Args:
        face_rgb : HxWx3 uint8 RGB image (aligned, 224×224)
        wavelet  : mother wavelet — 'haar' has compact support, ideal for edges
        level    : decomposition depth (paper uses level=1 for simplicity)

    Returns:
        HxWx3 float32 array in [0, 1] ready for ViT normalisation transforms.
    """
    # Step 1 — Grayscale conversion (SWT operates on single channel)
    gray = cv2.cvtColor(face_rgb, cv2.COLOR_RGB2GRAY).astype(np.float64)

    # Step 2 — 2-D Stationary (undecimated) Wavelet Transform
    # pywt.swt2 returns [(cA_n, (cH_n, cV_n, cD_n)), ..., (cA_1, (cH_1, cV_1, cD_1))]
    # coarsest level first.  With level=1: coeffs[0] is the only element.
    coeffs = pywt.swt2(gray, wavelet=wavelet, level=level)
    _, (cH, cV, cD) = coeffs[0]   # discard approximation (cA)

    # Step 3 — Combined high-frequency magnitude (paper §3.1)
    #   C = sqrt(cH² + cV² + cD²)
    C = np.sqrt(cH**2 + cV**2 + cD**2).astype(np.float32)

    # Step 4 — Min-max normalise C to [0, 1]
    c_min, c_max = C.min(), C.max()
    C = (C - c_min) / (c_max - c_min + 1e-8)

    # Step 5 — Repeat into 3 identical channels so ViT sees (H, W, 3)
    #   np.stack([C, C, C], axis=-1) → shape (224, 224, 3)
    three_ch = np.stack([C, C, C], axis=-1)   # float32, values in [0, 1]

    return three_ch


print('SWT function ready (combined magnitude C = sqrt(H²+V²+D²), 3-channel repeat).')


## Cell 8 - PyTorch Dataset Class

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Augmentation helper ────────────────────────────────────────────────────
import torchvision.transforms.functional as TF
import random

class RandomRotate90:
    """Randomly rotate image by exactly 90 degrees."""
    def __call__(self, img):
        if random.random() > 0.5:
            return TF.rotate(img, 90)
        return img

# ── TRAIN transforms — augmentation applied here ──────────────────────────
TRAIN_TRANSFORMS = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),        # augmentation 1: flip
    RandomRotate90(),                               # augmentation 2: 90° rotate
    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=5, sigma=(0.1, 2.0))
    ], p=0.2),                                      # augmentation 3: blur
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.2, contrast=0.2, saturation=0.1)
    ], p=0.3),                                      # keeping existing jitter
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── EVAL transforms — NO augmentation ever ────────────────────────────────
EVAL_TRANSFORMS = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('✅ Augmentations set:')
print('   1. Random horizontal flip  (p=0.50)')
print('   2. Random 90° rotation     (p=0.50)')
print('   3. Gaussian blur           (p=0.40)')
print('   4. Color jitter            (p=0.30)')


def apply_swt(face_rgb, wavelet='haar', level=1):
    """
    SWT per SWTVIT paper §3.1:
        C = sqrt(H² + V² + D²)
    Returns HxWx3 float32 in [0,1] (C repeated 3 times for ViT input).
    """
    gray   = cv2.cvtColor(face_rgb, cv2.COLOR_RGB2GRAY).astype(np.float64)
    coeffs = pywt.swt2(gray, wavelet=wavelet, level=level)
    _, (cH, cV, cD) = coeffs[0]

    C = np.sqrt(cH**2 + cV**2 + cD**2).astype(np.float32)
    c_min, c_max = C.min(), C.max()
    C = (C - c_min) / (c_max - c_min + 1e-8)

    return np.stack([C, C, C], axis=-1)   # (H, W, 3) float32 in [0,1]


class DeepfakeFrameDataset(Dataset):
    """
    Loads pre-saved face PNG crops from disk, applies SWT, then transforms.

    Pipeline per worker (CPU only — safe for num_workers > 0):
        Load PNG  →  apply_swt  →  PIL convert  →  transform  →  (tensor, label)

    No MTCNN, no CUDA inside workers.  Face detection was already done by
    pre_extract_faces() in the main process.
    """

    def __init__(
        self,
        face_label_pairs: List[Tuple[str, int]],
        transform=None,
        wavelet: str = 'haar',
        swt_level: int = 1,
    ):
        self.pairs     = face_label_pairs
        self.transform = transform
        self.wavelet   = wavelet
        self.swt_level = swt_level

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        face_path, label = self.pairs[idx]

        # ── Load pre-saved face crop ──────────────────────────────────────
        img_bgr = cv2.imread(face_path)
        if img_bgr is None:
            return None   # corrupted file — safe_collate will drop it
        face_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # ── Apply SWT ────────────────────────────────────────────────────
        swt_img = apply_swt(face_rgb, wavelet=self.wavelet, level=self.swt_level)

        # ── Convert to PIL and apply transforms ──────────────────────────
        pil = Image.fromarray((swt_img * 255).astype(np.uint8))
        tensor = self.transform(pil) if self.transform else transforms.ToTensor()(pil)

        return tensor, label


def safe_collate(batch):
    """Drop None items (corrupted files). Return None if entire batch is empty."""
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    return torch.utils.data.dataloader.default_collate(batch)


# ── Instantiate datasets from pre-extracted face pairs ────────────────────
train_dataset = DeepfakeFrameDataset(
    TRAIN_FACES, transform=TRAIN_TRANSFORMS,
    wavelet=CFG['WAVELET'], swt_level=CFG['SWT_LEVEL'],
)
val_dataset = DeepfakeFrameDataset(
    VAL_FACES, transform=EVAL_TRANSFORMS,
    wavelet=CFG['WAVELET'], swt_level=CFG['SWT_LEVEL'],
)
test_dataset = DeepfakeFrameDataset(
    TEST_FACES, transform=EVAL_TRANSFORMS,
    wavelet=CFG['WAVELET'], swt_level=CFG['SWT_LEVEL'],
)

print(f'\n[Dataset sizes — post face-detection filter]')
print(f'  Train : {len(train_dataset):>6} face samples')
print(f'  Val   : {len(val_dataset):>6} face samples')
print(f'  Test  : {len(test_dataset):>6} face samples')

# ── Weighted sampler for class imbalance ──────────────────────────────────
label_counts   = np.bincount([lbl for _, lbl in TRAIN_FACES], minlength=CFG['NUM_CLASSES'])
class_weights  = 1.0 / (label_counts + 1e-8)
sample_weights = torch.tensor(
    [class_weights[lbl] for _, lbl in TRAIN_FACES], dtype=torch.float32
)
sampler = WeightedRandomSampler(
    sample_weights, num_samples=len(sample_weights), replacement=True
)

# ── DataLoaders — num_workers=2 is now safe (no CUDA in workers) ──────────
TRAIN_LOADER = DataLoader(
    train_dataset, batch_size=CFG['BATCH_SIZE'],
    sampler=sampler, num_workers=2, pin_memory=True, collate_fn=safe_collate,
)
VAL_LOADER = DataLoader(
    val_dataset, batch_size=CFG['BATCH_SIZE'],
    shuffle=False, num_workers=2, pin_memory=True, collate_fn=safe_collate,
)
TEST_LOADER = DataLoader(
    test_dataset, batch_size=CFG['BATCH_SIZE'],
    shuffle=False, num_workers=2, pin_memory=True, collate_fn=safe_collate,
)

print(f'\nDataLoaders ready.  Batch size: {CFG["BATCH_SIZE"]}')
print('Workers: 2 (safe — no CUDA inside workers)')




## Cell 9 - Vision Transformer (ViT) Model

In [ ]:

class DeepfakeViT(nn.Module):
    """
    Vision Transformer for deepfake detection (SWTVIT paper).

    Backbone : vit_small_patch16_224  (timm, pretrained on ImageNet-21k)
    Input    : 224×224 SWT-processed face images — 3 channels (C repeated)
    Output   : 4-class logits [real, deepfake, faceswap, face2face]

    ViT advantages for deepfake detection (from paper §3.3):
      • Self-attention captures long-range spatial inconsistencies in one layer
        without stacking many CNN layers to grow the receptive field.
      • Patch tokens (16×16) align naturally with local forgery artefact scales.
      • ImageNet pretraining provides strong low-level feature priors.
    """

    def __init__(
        self,
        model_name: str = 'vit_small_patch16_224',   # ← changed from vit_base
        num_classes: int = 4,
        pretrained: bool = True,
        dropout: float = 0.3,
    ):
        super().__init__()

        # Load pretrained ViT; strip its original classification head
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool='token',   # returns [CLS] token embedding
        )
        embed_dim = self.backbone.num_features   # 384 for ViT-Small, 768 for ViT-Base

        # Custom head: LayerNorm → Dropout → Linear
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (B, 3, 224, 224) normalised SWT images
        Returns:
            logits: (B, num_classes) raw unnormalised scores
        """
        features = self.backbone(x)   # (B, embed_dim) — [CLS] token
        return self.head(features)


# Instantiate model using CFG so the name can be overridden from config
MODEL = DeepfakeViT(
    model_name=CFG['MODEL_NAME'],
    num_classes=CFG['NUM_CLASSES'],
    pretrained=True,
).to(DEVICE)

total_params     = sum(p.numel() for p in MODEL.parameters())
trainable_params = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
print(f'Model   : {CFG["MODEL_NAME"]}')
print(f'  Total params     : {total_params:,}')
print(f'  Trainable params : {trainable_params:,}')
print(f'  Embed dim        : {MODEL.backbone.num_features}')


## Cell 10 - Optimizer, Scheduler, and Loss

In [ ]:
# Label-smoothed cross-entropy — reduces overconfidence, improves calibration
CRITERION = nn.CrossEntropyLoss(label_smoothing=0.05)  # reduced from 0.1

backbone_params = list(MODEL.backbone.parameters())
head_params     = list(MODEL.head.parameters())

# AdamW with lower layer-wise LRs for stability
OPTIMIZER = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CFG['LR'] / 10},  # 1e-5
    {'params': head_params,     'lr': CFG['LR']},         # 1e-4
], weight_decay=1e-2)

# Warmup for 5 epochs → then Cosine Annealing
# Warmup prevents wild updates in early epochs (fixes zigzag)
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

warmup_scheduler = LinearLR(
    OPTIMIZER,
    start_factor=0.1,    # starts at LR × 0.1
    end_factor=1.0,      # gradually reaches full LR
    total_iters=5,       # over first 5 epochs
)
cosine_scheduler = CosineAnnealingLR(
    OPTIMIZER,
    T_max=CFG['EPOCHS'] - 5,   # remaining epochs after warmup
    eta_min=1e-6,
)
SCHEDULER = SequentialLR(
    OPTIMIZER,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[5],      # switch to cosine at epoch 5
)

print('Loss      : CrossEntropyLoss (label_smoothing=0.05)')
print('Optimizer : AdamW with layer-wise LR')
print(f'  Backbone LR : {CFG["LR"] / 10:.2e}')
print(f'  Head LR     : {CFG["LR"]:.2e}')
print('Scheduler : LinearWarmup (5 epochs) → CosineAnnealingLR')
print(f'  Warmup epochs : 5')
print(f'  Cosine epochs : {CFG["EPOCHS"] - 5}')

## Cell 11 - Training and Validation Loop Functions

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    epoch: int,
    max_grad_norm: float = 1.0,
) -> Tuple[float, float]:
    """
    Run one complete training epoch.

    Gradient clipping (max_grad_norm=1.0) is applied after each backward pass
    to prevent exploding gradients during ViT fine-tuning on small datasets.

    Returns:
        (mean_loss, accuracy) over all processed samples in this epoch.
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    skipped_batches = 0

    pbar = tqdm(loader, desc=f'  [Train] Epoch {epoch}', leave=False)
    for batch in pbar:
        if batch is None:   # entire mini-batch had no detected faces
            skipped_batches += 1
            continue

        images, labels = batch
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)                      # (B, num_classes)
        loss   = criterion(logits, labels)
        loss.backward()

        # Gradient clipping — prevents exploding gradients in ViT attention layers
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)

        pbar.set_postfix({
            'loss': f'{total_loss / max(total, 1):.4f}',
            'acc':  f'{correct / max(total, 1):.3f}',
        })

    if skipped_batches > 0:
        print(f'    [WARN] {skipped_batches} batches skipped (no faces detected).')
    if total == 0:
        print(f'    [ERROR] Epoch {epoch}: zero valid samples — check face detection.')

    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    desc: str = 'Val',
) -> Tuple[float, float, np.ndarray, np.ndarray]:
    """
    Evaluate model on a DataLoader.

    Returns:
        (mean_loss, accuracy, all_predictions, all_true_labels)
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=f'  [{desc}]', leave=False)
    for batch in pbar:
        if batch is None:
            continue

        images, labels = batch
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (
        total_loss / max(total, 1),
        correct / max(total, 1),
        np.array(all_preds),
        np.array(all_labels),
    )


def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    val_acc: float,
    path: str,
):
    """Persist model weights, optimizer state, epoch, and config.
    Also auto-backs up best model to Google Drive."""
    torch.save({
        'epoch':     epoch,
        'val_acc':   val_acc,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'cfg':       CFG,
    }, path)
    print(f'  Checkpoint saved → {path}')

    # ── Auto backup BEST model to Drive immediately ────────────────
    if 'best' in path:
        drive_path = '/content/drive/MyDrive/data/best_model.pth'
        shutil.copy(path, drive_path)
        print(f'  ✅ Drive backup  → {drive_path}')


print('Training helpers defined (gradient clipping max_norm=1.0 active).')



In [ ]:
import torch

#later use

checkpoint = torch.load(
    '/content/drive/MyDrive/data/best_model.pth',
    map_location='cpu' # Explicitly map to CPU, as GPU might be unavailable
)
MODEL.load_state_dict(checkpoint['model'])
print(f"✅ Model loaded successfully!")
print(f"   Epoch   : {checkpoint['epoch']}")
print(f"   Val acc : {checkpoint['val_acc']:.4f}")

In [ ]:
#running my personal videos

from google.colab import files
import shutil

print("Upload your REAL video...")
uploaded = files.upload()
real_filename = list(uploaded.keys())[0]
shutil.copy(f'/content/{real_filename}',
            '/content/drive/MyDrive/data/my_real_video.mp4')
print(f"✅ Real video saved: {real_filename}")

print("\nUpload your FACESWAP video...")
uploaded2 = files.upload()
fake_filename = list(uploaded2.keys())[0]
shutil.copy(f'/content/{fake_filename}',
            '/content/drive/MyDrive/data/my_faceswap_video.mp4')
print(f"✅ Faceswap video saved: {fake_filename}")

In [ ]:
# from pathlib import Path

# CORRESPONDING_VIDEOS = [
#     ('/content/drive/MyDrive/data/original_sequences/youtube/c23/videos/585.mp4',
#      0, 'REAL'),
#     ('/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos/585_599.mp4',
#      1, 'DEEPFAKE'),
#     ('/content/drive/MyDrive/data/manipulated_sequences/Face2Face/c23/videos/585_599.mp4',
#      3, 'FACE2FACE'),
#     ('/content/drive/MyDrive/data/manipulated_sequences/FaceSwap/c23/videos/585_599.mp4',
#      2, 'FACESWAP'),
# ]

# print("Running on FaceForensics corresponding videos...")
# ff_results = []
# for vid_path, gt_label, gt_name in CORRESPONDING_VIDEOS:
#     pred = process_video(vid_path, gt_label, gt_name)
#     ff_results.append((Path(vid_path).name, gt_name,
#                        CLASS_NAMES.get(pred, 'UNKNOWN')))

# print(f'\n{"="*55}')
# print('FACEFORENSICS CORRESPONDING VIDEO RESULTS')
# print(f'{"="*55}')
# for vid, gt, pred in ff_results:
#     match = "✅" if gt == pred else "❌"
#     print(f'  {vid:<20} {gt:<12} {pred:<12} {match}')

## Cell 12 - Main Training Loop (20 epochs + early stopping)

In [ ]:
best_val_acc  = 0.0
patience_ctr  = 0
history: Dict[str, List[float]] = defaultdict(list)

BEST_CKPT = os.path.join(CFG['CHECKPOINT_DIR'], 'best_model.pth')
LAST_CKPT = os.path.join(CFG['CHECKPOINT_DIR'], 'last_model.pth')

print(f'Starting training for up to {CFG["EPOCHS"]} epochs ...')
print(f'Early stopping patience: {CFG["PATIENCE"]} epochs')
print('-' * 70)

for epoch in range(1, CFG['EPOCHS'] + 1):

    # Training pass
    train_loss, train_acc = train_one_epoch(
        MODEL, TRAIN_LOADER, OPTIMIZER, CRITERION, DEVICE, epoch
    )

    # Validation pass (used for early stopping + checkpoint selection)
    val_loss, val_acc, val_preds, val_labels = evaluate(
        MODEL, VAL_LOADER, CRITERION, DEVICE, desc='Val'
    )

    # Step cosine annealing scheduler
    SCHEDULER.step()

    # Record history for plotting
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    curr_lr = OPTIMIZER.param_groups[1]['lr']
    print(
        f'Epoch {epoch:>2}/{CFG["EPOCHS"]} | '
        f'train_loss={train_loss:.4f}  train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f} | '
        f'lr={curr_lr:.2e}'
    )

    # Save best checkpoint when validation accuracy improves
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_ctr = 0
        save_checkpoint(MODEL, OPTIMIZER, epoch, val_acc, BEST_CKPT)
    else:
        patience_ctr += 1
        print(f'  No improvement for {patience_ctr}/{CFG["PATIENCE"]} epochs')

    # Always save the most recent checkpoint
    save_checkpoint(MODEL, OPTIMIZER, epoch, val_acc, LAST_CKPT)

    # Early stopping: terminate if no improvement for PATIENCE consecutive epochs
    if patience_ctr >= CFG['PATIENCE']:
        print(f'\nEarly stopping triggered at epoch {epoch}.')
        break

print(f'\nTraining complete. Best val acc: {best_val_acc:.4f}')

In [ ]:
from pathlib import Path

frame_base = Path('/content/ff_frames')
face_base  = Path('/content/ff_faces')

print("=== FRAME COUNTS ===")
for label in range(4):
    folder = frame_base / f'label_{label}'
    if folder.exists():
        frames = list(folder.rglob('frame_*.png'))
        print(f"  label_{label}: {len(frames)} frames")
    else:
        print(f"  label_{label}: ❌ folder missing!")

print("\n=== FACE COUNTS ===")
for label in range(4):
    folder = face_base / f'label_{label}'
    if folder.exists():
        faces = list(folder.rglob('*.png'))
        print(f"  label_{label}: {len(faces)} faces")
    else:
        print(f"  label_{label}: ❌ folder missing!")

## Cell 13 - Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = len(history['train_loss'])
xs = range(1, epochs_ran + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves -- Deepfake Detection ViT', fontsize=14, fontweight='bold')

ax1.plot(xs, history['train_loss'], label='Train', marker='o')
ax1.plot(xs, history['val_loss'],   label='Val',   marker='s')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(xs, history['train_acc'], label='Train', marker='o')
ax2.plot(xs, history['val_acc'],   label='Val',   marker='s')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CFG['CHECKPOINT_DIR'], 'training_curves.png'), dpi=150)
plt.show()
print('Curves saved.')

## Cell 14 - Frame-Level Test Evaluation

In [ ]:
# Load best checkpoint (selected on validation accuracy)
checkpoint = torch.load(BEST_CKPT, map_location=DEVICE)
MODEL.load_state_dict(checkpoint['model'])
print(f'Loaded best model from epoch {checkpoint["epoch"]} (val_acc={checkpoint["val_acc"]:.4f})')

# Run frame-level test evaluation
test_loss, test_acc, test_preds, test_labels = evaluate(
    MODEL, TEST_LOADER, CRITERION, DEVICE, desc='Test'
)

print(f'\n=======================================')
print(f'  FRAME-LEVEL TEST RESULTS')
print(f'=======================================')
print(f'  Test Loss     : {test_loss:.4f}')
print(f'  Test Accuracy : {test_acc:.4f}')

label_names = [CFG['LABEL_MAP'][i] for i in range(CFG['NUM_CLASSES'])]

precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, average=None, labels=list(range(CFG['NUM_CLASSES']))
)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='macro'
)

print(f'\n  Per-class metrics:')
print(f'  {"Class":<12} {"Precision":>10} {"Recall":>8} {"F1":>8} {"Support":>9}')
print('  ' + '-' * 50)
for i, name in enumerate(label_names):
    print(f'  {name:<12} {precision[i]:>10.4f} {recall[i]:>8.4f} {f1[i]:>8.4f} {int(support[i]):>9}')
print('  ' + '-' * 50)
print(f'  {"MACRO avg":<12} {macro_p:>10.4f} {macro_r:>8.4f} {macro_f1:>8.4f}')
print('\n  Full classification report:')
print(classification_report(test_labels, test_preds, target_names=label_names))

## Cell 15 - Video-Level Prediction (Averaging Frame Probabilities)

Video-level prediction aggregates softmax probabilities across all valid frames
of a video and takes the argmax of the average. This is more robust than
frame-level voting because it weights confident frames more than uncertain ones.

In [ ]:

@torch.no_grad()
def predict_video(
    video_path: str,
    model: nn.Module,
    detector: MTCNN,
    transform,
    device: torch.device,
    fps: float = 0.2,
    max_frames: int = 30,
) -> Optional[int]:
    """
    Predict the class for an entire video (SWTVIT pipeline):
      1. Extract frames (ffmpeg @ fps)
      2. For each frame: detect face → apply SWT → run ViT
      3. Accumulate softmax probabilities across valid frames
      4. Return argmax of the averaged probability vector

    Frames with no detectable face are silently skipped (no fallback).

    Returns:
        Predicted class index, or None if no valid frames were found.
    """
    model.eval()

    tmp_dir = f'/tmp/vid_{Path(video_path).stem}'
    frames  = extract_frames(video_path, tmp_dir, fps, max_frames)

    if not frames:
        return None   # corrupted or empty video

    prob_sum    = None
    valid_count = 0

    for fp in frames:
        # ── SWTVIT pipeline: Frame → Face → SWT → ViT ───
        face = detect_and_align(fp, detector, CFG['FACE_SIZE'])
        if face is None:
            continue   # no face — skip frame, never substitute a crop

        swt_img = apply_swt(face, wavelet=CFG['WAVELET'], level=CFG['SWT_LEVEL'])
        pil     = Image.fromarray((swt_img * 255).astype(np.uint8))
        tensor  = transform(pil).unsqueeze(0).to(device)

        logits = model(tensor)                    # (1, num_classes)
        probs  = F.softmax(logits, dim=1)         # (1, num_classes)

        prob_sum     = probs if prob_sum is None else prob_sum + probs
        valid_count += 1

    shutil.rmtree(tmp_dir, ignore_errors=True)

    if valid_count == 0:
        return None

    avg_probs  = prob_sum / valid_count
    prediction = int(avg_probs.argmax(dim=1).item())
    return prediction


def evaluate_video_level(
    video_label_list: List[Tuple[str, int]],
    model: nn.Module,
    detector: MTCNN,
    transform,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Run video-level evaluation over a list of (video_path, label) pairs.
    Videos where no faces are found are silently skipped.

    Returns:
        (predictions, ground_truths) as numpy arrays.
    """
    preds, gts = [], []
    for vid_path, gt_label in tqdm(video_label_list, desc='Video-level eval'):
        pred = predict_video(vid_path, model, detector, transform, device)
        if pred is None:
            continue
        preds.append(pred)
        gts.append(gt_label)

    return np.array(preds), np.array(gts)


print('Running VIDEO-LEVEL evaluation on test set ...')
vid_preds, vid_gts = evaluate_video_level(
    TEST_VIDS, MODEL, MTCNN_DETECTOR, EVAL_TRANSFORMS, DEVICE
)

label_names = [CFG['LABEL_MAP'][i] for i in range(CFG['NUM_CLASSES'])]
vid_acc = accuracy_score(vid_gts, vid_preds)
v_p, v_r, v_f1, v_sup = precision_recall_fscore_support(
    vid_gts, vid_preds, average=None, labels=list(range(CFG['NUM_CLASSES']))
)
vm_p, vm_r, vm_f1, _ = precision_recall_fscore_support(
    vid_gts, vid_preds, average='macro'
)

print(f'\n{"="*45}')
print(f'  VIDEO-LEVEL TEST RESULTS')
print(f'{"="*45}')
print(f'  Videos evaluated : {len(vid_gts)}')
print(f'  Accuracy         : {vid_acc:.4f}')
print(f'\n  Per-class metrics:')
print(f'  {"Class":<12} {"Precision":>10} {"Recall":>8} {"F1":>8} {"Support":>9}')
print('  ' + '-' * 50)
for i, name in enumerate(label_names):
    print(f'  {name:<12} {v_p[i]:>10.4f} {v_r[i]:>8.4f} {v_f1[i]:>8.4f} {int(v_sup[i]):>9}')
print('  ' + '-' * 50)
print(f'  {"MACRO avg":<12} {vm_p:>10.4f} {vm_r:>8.4f} {vm_f1:>8.4f}')
print(classification_report(vid_gts, vid_preds, target_names=label_names))


## Cell 16 - Confusion Matrix Visualization

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

def plot_confusion_matrix(y_true, y_pred, class_names, title='Confusion Matrix', save_path=None):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    for ax, data, fmt, subtitle in zip(
        axes,
        [cm, cm_norm],
        ['d', '.2f'],
        ['Raw Counts', 'Normalized'],
    ):
        im = ax.imshow(data, cmap='Blues')
        ax.set_title(subtitle)
        ax.set_xticks(range(len(class_names)))
        ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels(class_names, rotation=30, ha='right')
        ax.set_yticklabels(class_names)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        plt.colorbar(im, ax=ax)

        thresh = data.max() / 2
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                ax.text(j, i, format(data[i, j], fmt),
                        ha='center', va='center',
                        color='white' if data[i, j] > thresh else 'black',
                        fontsize=10)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


# Frame-level
plot_confusion_matrix(
    test_labels, test_preds, label_names,
    title='Frame-Level Confusion Matrix',
    save_path=os.path.join(CFG['CHECKPOINT_DIR'], 'cm_frame.png'),
)

# Video-level
if len(vid_gts) > 0:
    plot_confusion_matrix(
        vid_gts, vid_preds, label_names,
        title='Video-Level Confusion Matrix',
        save_path=os.path.join(CFG['CHECKPOINT_DIR'], 'cm_video.png'),
    )

print('Confusion matrices saved.')

## Cell 17 - Single Video Inference (Demo)

In [ ]:

def infer_single_video(
    video_path: str,
    model: nn.Module,
    detector: MTCNN,
    transform,
    device: torch.device,
) -> Dict:
    """
    Run the full SWTVIT pipeline on a single video and return a result dict.

    Returns dict with keys:
        video       : input path
        prediction  : class name string
        class_idx   : integer class index
    Or {'error': ...} if no valid frames were found.
    """
    pred_idx = predict_video(video_path, model, detector, transform, device)
    if pred_idx is None:
        return {'error': 'No valid faces detected', 'video': video_path}

    return {
        'video':      video_path,
        'prediction': CFG['LABEL_MAP'][pred_idx],
        'class_idx':  pred_idx,
    }


# Demo: run on the first test video
if TEST_VIDS:
    sample_vid, sample_lbl = TEST_VIDS[0]
    print(f'Inferring : {sample_vid}')
    print(f'  Ground truth : {CFG["LABEL_MAP"][sample_lbl]}')

    result = infer_single_video(
        sample_vid, MODEL, MTCNN_DETECTOR, EVAL_TRANSFORMS, DEVICE
    )
    print(f'  Prediction   : {result.get("prediction", result.get("error"))}')
else:
    print('No test videos found — skipping demo.')





## Cell 18 - Export and Final Summary

In [ ]:
# Save training history as JSON for later analysis
history_path = os.path.join(CFG['CHECKPOINT_DIR'], 'history.json')
with open(history_path, 'w') as f:
    json.dump(dict(history), f, indent=2)

# Save config
cfg_path = os.path.join(CFG['CHECKPOINT_DIR'], 'config.json')
with open(cfg_path, 'w') as f:
    # LABEL_MAP keys are ints; convert to str for JSON serialization
    cfg_serializable = {k: (str(v) if isinstance(v, dict) else v) for k, v in CFG.items()}
    json.dump(cfg_serializable, f, indent=2)

# Final summary
print('=' * 62)
print('  FINAL PIPELINE SUMMARY')
print('=' * 62)
print(f'  Model              : {CFG["MODEL_NAME"]}')
print(f'  Trainable params   : {trainable_params:,}')
print(f'  Frame extraction   : {CFG["FPS"]} fps  (1 frame / 5 s)')
print(f'  Face size          : {CFG["FACE_SIZE"]} x {CFG["FACE_SIZE"]}')
print(f'  SWT wavelet        : {CFG["WAVELET"]}  (level {CFG["SWT_LEVEL"]})')
print(f'  Batch size         : {CFG["BATCH_SIZE"]}')
print(f'  Best val acc       : {best_val_acc:.4f}')
print(f'  Test acc (frame)   : {test_acc:.4f}')
if len(vid_gts) > 0:
    print(f'  Test acc (video)   : {vid_acc:.4f}')
print(f'  Macro F1 (frame)   : {macro_f1:.4f}')
print(f'  Checkpoints        : {CFG["CHECKPOINT_DIR"]}')
print('=' * 62)
print('Pipeline complete!')

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# ── 1. Check how many frames were extracted ────────────────────────────────
print("=== FRAME COUNTS ===")
frame_base = Path('/content/ff_frames')

if not frame_base.exists():
    print("❌ /content/ff_frames does not exist — frames not yet extracted!")
else:
    for label_folder in sorted(frame_base.iterdir()):
        frames = list(label_folder.rglob('frame_*.png'))
        print(f"  {label_folder.name}: {len(frames)} frames")

# ── 2. Check how many face crops were extracted ────────────────────────────
print("\n=== FACE CROP COUNTS ===")
face_base = Path('/content/ff_faces')

if not face_base.exists():
    print("❌ /content/ff_faces does not exist — face detection not yet run!")
else:
    for label_folder in sorted(face_base.iterdir()):
        faces = list(label_folder.rglob('*.png'))
        print(f"  {label_folder.name}: {len(faces)} face crops")

# ── 3. Display sample RAW FRAMES (before face detection) ──────────────────
print("\n=== SAMPLE RAW FRAMES ===")
all_frames = list(frame_base.rglob('frame_*.png')) if frame_base.exists() else []

if not all_frames:
    print("No frames found!")
else:
    # Pick one frame from each label folder
    samples = {}
    for f in all_frames:
        label = f.parts[f.parts.index('ff_frames') + 1]  # e.g. label_0
        if label not in samples:
            samples[label] = f

    fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5))
    if len(samples) == 1:
        axes = [axes]

    for ax, (label, fpath) in zip(axes, sorted(samples.items())):
        img = cv2.imread(str(fpath))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{label}\n{fpath.name}", fontsize=9)
        ax.axis('off')

    plt.suptitle("Sample RAW Frames (one per label)", fontweight='bold')
    plt.tight_layout()
    plt.show()

# ── 4. Display sample FACE CROPS (after MTCNN) ────────────────────────────
print("\n=== SAMPLE FACE CROPS ===")
all_faces = list(face_base.rglob('*.png')) if face_base.exists() else []

if not all_faces:
    print("No face crops found!")
else:
    samples_face = {}
    for f in all_faces:
        label = f.parts[f.parts.index('ff_faces') + 1]
        if label not in samples_face:
            samples_face[label] = f

    fig, axes = plt.subplots(1, len(samples_face), figsize=(5 * len(samples_face), 5))
    if len(samples_face) == 1:
        axes = [axes]

    for ax, (label, fpath) in zip(axes, sorted(samples_face.items())):
        img = cv2.imread(str(fpath))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{label}\n{fpath.name}", fontsize=9)
        ax.axis('off')

    plt.suptitle("Sample FACE CROPS after MTCNN (one per label)", fontweight='bold')
    plt.tight_layout()
    plt.show()

# ── 5. Display a STRIP of frames from ONE video ───────────────────────────
print("\n=== FRAME STRIP FROM ONE VIDEO ===")
if all_frames:
    # Pick the first video folder found
    first_video_folder = None
    for label_folder in sorted(frame_base.iterdir()):
        for vid_folder in sorted(label_folder.iterdir()):
            if vid_folder.is_dir():
                first_video_folder = vid_folder
                break
        if first_video_folder:
            break

    if first_video_folder:
        vid_frames = sorted(first_video_folder.glob('frame_*.png'))
        print(f"Video: {first_video_folder.name}  ({len(vid_frames)} frames)")

        # Show up to 8 frames in a row
        show = vid_frames[::max(1, len(vid_frames)//8)][:8]
        fig, axes = plt.subplots(1, len(show), figsize=(3 * len(show), 3))
        if len(show) == 1:
            axes = [axes]

        for ax, fp in zip(axes, show):
            img = cv2.imread(str(fp))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(fp.name, fontsize=7)
            ax.axis('off')

        plt.suptitle(f"Frame strip — {first_video_folder.name}", fontweight='bold')
        plt.tight_layout()
        plt.show()

In [ ]:
import os
ckpt = '/content/checkpoints/best_model.pth'
if os.path.exists(ckpt):
    print("✅ Checkpoint exists!")
else:
    print("❌ Checkpoint missing — training needs to be rerun")

In [ ]:
# Save checkpoint to Drive so it's safe
import shutil
shutil.copy(
    '/content/checkpoints/best_model.pth',
    '/content/drive/MyDrive/data/best_model.pth'
)
print("✅ Checkpoint backed up to Drive!")

In [ ]:
# running this to see what vidoes i have
from pathlib import Path

# Show 2 videos — one real, one deepfake
real_videos = list(Path('/content/drive/MyDrive/data/original_sequences/youtube/c23/videos').glob('*.mp4'))
fake_videos = list(Path('/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos').glob('*.mp4'))

print(f"Total real videos: {len(real_videos)}")
print(f"Total deepfake videos: {len(fake_videos)}")

print("\nFirst 3 real videos:")
for v in real_videos[:3]:
    print(f"  {v}")

print("\nFirst 3 deepfake videos:")
for v in fake_videos[:3]:
    print(f"  {v}")

In [ ]:

# segmentation
import cv2
import torch
import numpy as np
import subprocess
import shutil
from pathlib import Path
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F

# ── Video paths ────────────────────────────────────────────────────────────
TEST_VIDEOS = [
    ('/content/drive/MyDrive/data/original_sequences/youtube/c23/videos/572.mp4', 0, 'REAL'),
    ('/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos/360_437.mp4', 1, 'DEEPFAKE'),
]

# ── GradCAM for ViT ────────────────────────────────────────────────────────
class ViTGradCAM:
    def __init__(self, model):
        self.model     = model
        self.gradients = None
        self.activations= None
        # Hook onto the LAST attention block
        target_layer = model.backbone.blocks[-1].norm1
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, tensor):
        """Returns a 224x224 heatmap numpy array (0-255 uint8)."""
        self.model.zero_grad()
        logits = self.model(tensor)
        pred   = logits.argmax(dim=1)
        logits[0, pred].backward()

        # Average gradients across tokens
        weights = self.gradients.mean(dim=1, keepdim=True)  # (1, 1, dim)
        cam     = (weights * self.activations).sum(dim=-1)   # (1, tokens)
        cam     = cam.squeeze(0)[1:]                          # remove CLS token

        # Reshape patch tokens back to 2D grid (14x14 for ViT-Small patch16)
        grid_size = int(cam.shape[0] ** 0.5)
        cam = cam.reshape(grid_size, grid_size).cpu().numpy()
        cam = np.maximum(cam, 0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam, (224, 224))
        return cam, int(pred.item())


# ── Colour coding per class ────────────────────────────────────────────────
CLASS_COLORS = {
    0: (0, 255, 0),    # REAL       → Green
    1: (0, 0, 255),    # DEEPFAKE   → Red
    2: (255, 0, 0),    # FACESWAP   → Blue
    3: (255, 165, 0),  # FACE2FACE  → Orange
}
CLASS_NAMES = {0: 'REAL', 1: 'DEEPFAKE', 2: 'FACESWAP', 3: 'FACE2FACE'}


def process_video(video_path, ground_truth_label, ground_truth_name):
    print(f'\n{"="*55}')
    print(f'Processing: {Path(video_path).name}')
    print(f'Ground truth: {ground_truth_name}')
    print(f'{"="*55}')

    # ── Step 1: Extract frames ─────────────────────────────────────────
    tmp_frames = f'/tmp/test_frames_{Path(video_path).stem}'
    shutil.rmtree(tmp_frames, ignore_errors=True)
    os.makedirs(tmp_frames, exist_ok=True)

    subprocess.run([
        'ffmpeg', '-y', '-i', video_path,
        '-vf', 'fps=1',          # 1 frame per second for testing
        '-q:v', '2',
        f'{tmp_frames}/frame_%04d.png'
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    frames = sorted(Path(tmp_frames).glob('frame_*.png'))
    print(f'  Frames extracted: {len(frames)}')

    if not frames:
        print('  ❌ No frames extracted!')
        return

    # ── Step 2: Set up GradCAM ────────────────────────────────────────
    MODEL.eval()
    gradcam = ViTGradCAM(MODEL)

    # ── Step 3: Process each frame ────────────────────────────────────
    annotated_dir = f'/tmp/annotated_{Path(video_path).stem}'
    shutil.rmtree(annotated_dir, ignore_errors=True)
    os.makedirs(annotated_dir, exist_ok=True)

    frame_predictions = []

    for i, frame_path in enumerate(frames):
        # Load frame
        img_bgr = cv2.imread(str(frame_path))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = img_rgb.shape[:2]

        # Detect face
        face = detect_and_align(str(frame_path), MTCNN_DETECTOR, CFG['FACE_SIZE'])
        if face is None:
            # No face — write original frame with "NO FACE" text
            out_frame = img_bgr.copy()
            cv2.putText(out_frame, 'NO FACE DETECTED', (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)
            cv2.imwrite(f'{annotated_dir}/frame_{i:04d}.png', out_frame)
            continue

        # Apply SWT
        swt_img = apply_swt(face, wavelet=CFG['WAVELET'], level=CFG['SWT_LEVEL'])
        pil     = Image.fromarray((swt_img * 255).astype(np.uint8))
        tensor  = EVAL_TRANSFORMS(pil).unsqueeze(0).to(DEVICE)
        tensor.requires_grad_(True)

        # GradCAM
        cam, pred_class = gradcam.generate(tensor)
        frame_predictions.append(pred_class)

        color     = CLASS_COLORS[pred_class]
        label_txt = CLASS_NAMES[pred_class]

        # ── Overlay heatmap on face region ────────────────────────────
        heatmap     = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
        heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        face_blend  = cv2.addWeighted(face, 0.6, heatmap_rgb, 0.4, 0)

        # Find bounding box from high-activation region of CAM
        thresh = (cam > 0.5).astype(np.uint8)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        annotated = img_rgb.copy()

        # Detect face box location in original frame
        from facenet_pytorch import MTCNN as _MTCNN
        pil_orig = Image.fromarray(img_rgb)
        try:
            boxes, probs, _ = MTCNN_DETECTOR.detect(pil_orig, landmarks=True)
        except:
            boxes = None

        if boxes is not None and len(boxes) > 0:
            box = boxes[0].astype(int)
            fx1 = max(0, box[0]); fy1 = max(0, box[1])
            fx2 = min(orig_w, box[2]); fy2 = min(orig_h, box[3])
            face_w = fx2 - fx1
            face_h = fy2 - fy1

            # Place blended face back
            face_resized_back = cv2.resize(face_blend, (face_w, face_h))
            annotated[fy1:fy2, fx1:fx2] = face_resized_back

            # Draw bounding box around face
            cv2.rectangle(annotated, (fx1, fy1), (fx2, fy2), color, 3)

            # Draw GradCAM-based inner box (tampered region)
            if contours:
                cx, cy, cw, ch = cv2.boundingRect(max(contours, key=cv2.contourArea))
                # Scale contour coords to face region in original frame
                sx = face_w / 224; sy = face_h / 224
                ix1 = fx1 + int(cx * sx); iy1 = fy1 + int(cy * sy)
                ix2 = fx1 + int((cx+cw)*sx); iy2 = fy1 + int((cy+ch)*sy)
                cv2.rectangle(annotated, (ix1, iy1), (ix2, iy2), color, 2)

        # ── Draw label banner ─────────────────────────────────────────
        banner_h = 50
        banner   = np.zeros((banner_h, orig_w, 3), dtype=np.uint8)
        banner[:] = color
        gt_match  = '>> CORRECT' if pred_class == ground_truth_label else '>> WRONG'
        txt = f'PREDICTION: {label_txt}   GT: {ground_truth_name}   {gt_match}   Frame {i+1}/{len(frames)}'
        cv2.putText(banner, txt, (10, 33),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        final_frame = np.vstack([banner, annotated])
        out_path    = f'{annotated_dir}/frame_{i:04d}.png'
        cv2.imwrite(out_path, cv2.cvtColor(final_frame, cv2.COLOR_RGB2BGR))

    # ── Step 4: Stitch frames into output video ────────────────────────
    out_video = f'/content/output_{Path(video_path).stem}_annotated.mp4'
    subprocess.run([
        'ffmpeg', '-y',
        '-framerate', '5',
        '-i', f'{annotated_dir}/frame_%04d.png',
        '-c:v', 'libx264',
        '-pix_fmt', 'yuv420p',
        out_video
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # ── Step 5: Summary ────────────────────────────────────────────────
    from collections import Counter
    vote = Counter(frame_predictions).most_common(1)
    final_pred = vote[0][0] if vote else -1
    print(f'  Frame predictions : {[CLASS_NAMES[p] for p in frame_predictions]}')
    print(f'  Final prediction  : {CLASS_NAMES.get(final_pred, "UNKNOWN")}')
    print(f'  Ground truth      : {ground_truth_name}')
    print(f'  Output video      : {out_video}')

    # Copy to Drive
    drive_out = f'/content/drive/MyDrive/data/output_{Path(video_path).stem}_annotated.mp4'
    shutil.copy(out_video, drive_out)
    print(f'  Saved to Drive    : {drive_out}')

    # Cleanup temp folders
    shutil.rmtree(tmp_frames, ignore_errors=True)
    shutil.rmtree(annotated_dir, ignore_errors=True)

    return final_pred


# ── Run on both test videos ────────────────────────────────────────────────
import os
results = []
for vid_path, gt_label, gt_name in TEST_VIDEOS:
    pred = process_video(vid_path, gt_label, gt_name)
    results.append((Path(vid_path).name, gt_name, CLASS_NAMES.get(pred, 'UNKNOWN')))

print(f'\n{"="*55}')
print('FINAL RESULTS SUMMARY')
print(f'{"="*55}')
print(f'  {"Video":<20} {"Ground Truth":<15} {"Predicted":<15}')
print(f'  {"-"*50}')
for vid, gt, pred in results:
    match = "✅" if gt == pred else "❌"
    print(f'  {vid:<20} {gt:<15} {pred:<15} {match}')
print(f'\nOutput videos saved to Google Drive!')

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/data/output_572_annotated.mp4')
files.download('/content/drive/MyDrive/data/output_360_437_annotated.mp4')

In [ ]:
#running this to see what all files i have
from pathlib import Path

f2f_dir  = Path('/content/drive/MyDrive/data/manipulated_sequences/Face2Face/c23/videos')
swap_dir = Path('/content/drive/MyDrive/data/manipulated_sequences/FaceSwap/c23/videos')

f2f_videos  = list(f2f_dir.glob('*.mp4'))
swap_videos = list(swap_dir.glob('*.mp4'))

print(f"Face2Face videos found : {len(f2f_videos)}")
print("\nFirst 3 Face2Face:")
for v in f2f_videos[:3]:
    print(f"  {v.name}")

print(f"\nFaceSwap videos found : {len(swap_videos)}")
print("\nFirst 3 FaceSwap:")
for v in swap_videos[:3]:
    print(f"  {v.name}")

In [ ]:
# ── Test videos — 2 Face2Face + 2 FaceSwap ────────────────────────────────
TEST_VIDEOS_NEW = [
    (
        '/content/drive/MyDrive/data/original_sequences/youtube/c23/videos/585.mp4',
        0, 'REAL'
    ),
    (
        '/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos/585_599.mp4',
        1, 'DEEPFAKE'
    ),
    (
        '/content/drive/MyDrive/data/manipulated_sequences/Face2Face/c23/videos/585_599.mp4',
        3, 'FACE2FACE'
    ),
    (
        '/content/drive/MyDrive/data/manipulated_sequences/FaceSwap/c23/videos/585_599.mp4',
        2, 'FACESWAP'
    ),
]

# ── Run on all 4 videos ────────────────────────────────────────────────────
results = []
for vid_path, gt_label, gt_name in TEST_VIDEOS_NEW:
    pred = process_video(vid_path, gt_label, gt_name)
    results.append((Path(vid_path).name, gt_name, CLASS_NAMES.get(pred, 'UNKNOWN')))

# ── Final summary ──────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('FINAL RESULTS SUMMARY — Face2Face + FaceSwap')
print(f'{"="*60}')
print(f'  {"Video":<20} {"Ground Truth":<15} {"Predicted":<15} {"Match"}')
print(f'  {"-"*55}')
for vid, gt, pred in results:
    match = "✅" if gt == pred else "❌"
    print(f'  {vid:<20} {gt:<15} {pred:<15} {match}')

In [ ]:
# ── Test videos — 2 Face2Face + 2 FaceSwap ────────────────────────────────
TEST_VIDEOS_NEW = [
    (
        '/content/drive/MyDrive/data/original_sequences/youtube/c23/videos/585.mp4',
        0, 'REAL'
    ),
    (
        '/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos/585_599.mp4',
        1, 'DEEPFAKE'
    ),
    (
        '/content/drive/MyDrive/data/manipulated_sequences/Face2Face/c23/videos/469_481.mp4',
        3, 'FACE2FACE'
    ),
    (
        '/content/drive/MyDrive/data/manipulated_sequences/FaceSwap/c23/videos/585_599.mp4',
        2, 'FACESWAP'
    ),
]

# ── Run on all 4 videos ────────────────────────────────────────────────────
results = []
for vid_path, gt_label, gt_name in TEST_VIDEOS_NEW:
    pred = process_video(vid_path, gt_label, gt_name)
    results.append((Path(vid_path).name, gt_name, CLASS_NAMES.get(pred, 'UNKNOWN')))

# ── Final summary ──────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('FINAL RESULTS SUMMARY — Face2Face + FaceSwap')
print(f'{"="*60}')
print(f'  {"Video":<20} {"Ground Truth":<15} {"Predicted":<15} {"Match"}')
print(f'  {"-"*55}')
for vid, gt, pred in results:
    match = "✅" if gt == pred else "❌"
    print(f'  {vid:<20} {gt:<15} {pred:<15} {match}')

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/data/output_585_annotated.mp4')
files.download('/content/drive/MyDrive/data/output_469_481_annotated.mp4')

In [ ]:
#running on my personal videos

MY_VIDEOS = [
    ('/content/drive/MyDrive/data/my_real_video.mp4',
     0, 'REAL'),
    ('/content/drive/MyDrive/data/my_faceswap_video.mp4',
     2, 'FACESWAP'),
]

print("Running on personal videos...")
my_results = []
for vid_path, gt_label, gt_name in MY_VIDEOS:
    pred = process_video(vid_path, gt_label, gt_name)
    my_results.append((Path(vid_path).name, gt_name,
                       CLASS_NAMES.get(pred, 'UNKNOWN')))

print(f'\n{"="*55}')
print('PERSONAL VIDEO RESULTS')
print(f'{"="*55}')
for vid, gt, pred in my_results:
    match = "✅" if gt == pred else "❌"
    print(f'  {vid:<20} {gt:<12} {pred:<12} {match}')

In [ ]:
#downlaoding the baove
from google.colab import files
import os

# List all annotated output videos
outputs = list(Path('/content/drive/MyDrive/data').glob('output_*_annotated.mp4'))
print(f"Output videos found: {len(outputs)}")
for v in outputs:
    print(f"  {v.name}")
    files.download(str(v))

In [ ]:
from pathlib import Path

# All test videos across all 4 classes
ALL_TEST_VIDEOS = (
    [(str(v), 0, 'REAL') for v in
     Path('/content/drive/MyDrive/data/original_sequences/youtube/c23/videos').glob('*.mp4')][:20] +
    [(str(v), 1, 'DEEPFAKE') for v in
     Path('/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos').glob('*.mp4')][:20] +
    [(str(v), 3, 'FACE2FACE') for v in
     Path('/content/drive/MyDrive/data/manipulated_sequences/Face2Face/c23/videos').glob('*.mp4')][:20] +
    [(str(v), 2, 'FACESWAP') for v in
     Path('/content/drive/MyDrive/data/manipulated_sequences/FaceSwap/c23/videos').glob('*.mp4')][:20]
)

print(f"Total videos to test: {len(ALL_TEST_VIDEOS)}")
print("Estimated time: ~20-30 mins on T4 GPU")

In [ ]:
import os
from collections import Counter
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import numpy as np

# ── Run evaluation on all 80 videos ───────────────────────────────────────
all_results = []
all_gt      = []
all_preds   = []

print(f'Running evaluation on {len(ALL_TEST_VIDEOS)} videos...')
print('This will take ~20-30 mins. Do NOT disconnect!\n')

for i, (vid_path, gt_label, gt_name) in enumerate(ALL_TEST_VIDEOS):
    print(f'[{i+1}/{len(ALL_TEST_VIDEOS)}] {Path(vid_path).name} | GT: {gt_name}')

    pred = process_video(vid_path, gt_label, gt_name)
    pred_name = CLASS_NAMES.get(pred, 'UNKNOWN')

    all_gt.append(gt_label)
    all_preds.append(pred if pred is not None else -1)
    all_results.append({
        'video'     : Path(vid_path).name,
        'gt_label'  : gt_label,
        'gt_name'   : gt_name,
        'pred_label': pred,
        'pred_name' : pred_name,
        'correct'   : pred == gt_label,
    })

    match = '✅' if pred == gt_label else '❌'
    print(f'   Predicted: {pred_name} {match}\n')

# ── Save results to Drive ──────────────────────────────────────────────────
import json
results_path = '/content/drive/MyDrive/data/evaluation_results.json'
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Results saved to: {results_path}')

# ── Overall accuracy ───────────────────────────────────────────────────────
valid_mask = [p != -1 for p in all_preds]
gt_valid   = [all_gt[i]    for i in range(len(all_gt))    if valid_mask[i]]
pred_valid = [all_preds[i] for i in range(len(all_preds)) if valid_mask[i]]

acc = accuracy_score(gt_valid, pred_valid)

print(f'\n{"="*60}')
print(f'  FULL EVALUATION RESULTS — 80 Videos')
print(f'{"="*60}')
print(f'  Videos evaluated : {len(gt_valid)}')
print(f'  Overall Accuracy : {acc:.4f} ({acc*100:.1f}%)')

# ── Per class accuracy ─────────────────────────────────────────────────────
print(f'\n  Per-class breakdown:')
label_names = ['REAL', 'DEEPFAKE', 'FACESWAP', 'FACE2FACE']
for label_idx, label_name in enumerate(label_names):
    cls_gt   = [g for g, p in zip(gt_valid, pred_valid) if g == label_idx]
    cls_pred = [p for g, p in zip(gt_valid, pred_valid) if g == label_idx]
    if cls_gt:
        cls_acc = sum(g == p for g, p in zip(cls_gt, cls_pred)) / len(cls_gt)
        print(f'  {label_name:<12}: {cls_acc*100:.1f}%  ({sum(g==p for g,p in zip(cls_gt,cls_pred))}/{len(cls_gt)} correct)')

# ── Classification report ──────────────────────────────────────────────────
print(f'\n  Full Classification Report:')
print(classification_report(
    gt_valid, pred_valid,
    target_names=label_names,
    labels=[0, 1, 2, 3]
))

# ── Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(gt_valid, pred_valid, labels=[0, 1, 2, 3])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Confusion Matrix — 80 Video Evaluation', fontsize=14, fontweight='bold')

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2f'],
    ['Raw Counts', 'Normalized']
):
    im = ax.imshow(data, cmap='Blues')
    ax.set_title(title, fontsize=12)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(label_names, rotation=30, ha='right')
    ax.set_yticklabels(label_names)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.colorbar(im, ax=ax)

    thresh = data.max() / 2
    for i in range(4):
        for j in range(4):
            ax.text(j, i, format(data[i,j], fmt),
                   ha='center', va='center', fontsize=11,
                   color='white' if data[i,j] > thresh else 'black')

plt.tight_layout()
cm_path = '/content/drive/MyDrive/data/confusion_matrix_80videos.png'
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Confusion matrix saved to: {cm_path}')

# ── Final summary table ────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('  VIDEO-BY-VIDEO SUMMARY')
print(f'{"="*60}')
print(f'  {"Video":<25} {"GT":<12} {"Predicted":<12} {"Match"}')
print(f'  {"-"*55}')
for r in all_results:
    match = '✅' if r['correct'] else '❌'
    print(f'  {r["video"]:<25} {r["gt_name"]:<12} {r["pred_name"]:<12} {match}')

In [ ]:
from pathlib import Path

REAL_VIDEO_ID = '572'

real_dir = Path('/content/drive/MyDrive/data/original_sequences/youtube/c23/videos')
deep_dir = Path('/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos')
f2f_dir  = Path('/content/drive/MyDrive/data/manipulated_sequences/Face2Face/c23/videos')
swap_dir = Path('/content/drive/MyDrive/data/manipulated_sequences/FaceSwap/c23/videos')

print(f'Corresponding videos for: {REAL_VIDEO_ID}\n')

print('REAL:')
for v in real_dir.glob(f'{REAL_VIDEO_ID}.mp4'):
    print(f'  {v}')

print('\nDEEPFAKE:')
for v in list(deep_dir.glob(f'{REAL_VIDEO_ID}_*.mp4')) + list(deep_dir.glob(f'*_{REAL_VIDEO_ID}.mp4')):
    print(f'  {v}')

print('\nFACE2FACE:')
for v in list(f2f_dir.glob(f'{REAL_VIDEO_ID}_*.mp4')) + list(f2f_dir.glob(f'*_{REAL_VIDEO_ID}.mp4')):
    print(f'  {v}')

print('\nFACESWAP:')
for v in list(swap_dir.glob(f'{REAL_VIDEO_ID}_*.mp4')) + list(swap_dir.glob(f'*_{REAL_VIDEO_ID}.mp4')):
    print(f'  {v}')

In [ ]:
from pathlib import Path

# Correct paths
real_dir = Path('/content/drive/MyDrive/data/original_sequences/youtube/c23/videos')
deep_dir = Path('/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos')
f2f_dir  = Path('/content/drive/MyDrive/data/manipulated_sequences/Face2Face/c23/videos')
swap_dir = Path('/content/drive/MyDrive/data/manipulated_sequences/FaceSwap/c23/videos')

# Show sample filenames from each folder
print("Sample REAL videos:")
for v in list(real_dir.glob('*.mp4'))[:5]:
    print(f"  {v.name}")

print("\nSample DEEPFAKE videos:")
for v in list(deep_dir.glob('*.mp4'))[:5]:
    print(f"  {v.name}")

print("\nSample FACE2FACE videos:")
for v in list(f2f_dir.glob('*.mp4'))[:5]:
    print(f"  {v.name}")

print("\nSample FACESWAP videos:")
for v in list(swap_dir.glob('*.mp4'))[:5]:
    print(f"  {v.name}")

In [ ]:
# Our corresponding video set!
CORRESPONDING_VIDEOS = [
    (
        '/content/drive/MyDrive/data/original_sequences/youtube/c23/videos/585.mp4',
        0, 'REAL'
    ),
    (
        '/content/drive/MyDrive/data/manipulated_sequences/Deepfakes/c23/videos/585_599.mp4',
        1, 'DEEPFAKE'
    ),
    (
        '/content/drive/MyDrive/data/manipulated_sequences/Face2Face/c23/videos/585_599.mp4',
        3, 'FACE2FACE'
    ),
    (
        '/content/drive/MyDrive/data/manipulated_sequences/FaceSwap/c23/videos/585_599.mp4',
        2, 'FACESWAP'
    ),
]

print("✅ Corresponding video set ready!")
print("\nVideos to process:")
for path, label, name in CORRESPONDING_VIDEOS:
    from pathlib import Path
    exists = Path(path).exists()
    print(f"  {name:<12} → {Path(path).name}  {'✅' if exists else '❌'}")

In [ ]:
import torch
from pathlib import Path

# Check exactly what model is saved on Drive
ckpt_path = '/content/drive/MyDrive/data/best_model.pth'

if Path(ckpt_path).exists():
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print(f"✅ Model found on Drive!")
    print(f"   Epoch    : {ckpt['epoch']}")
    print(f"   Val acc  : {ckpt['val_acc']:.4f}")
else:
    print("❌ No model found on Drive!")

# Also check checkpoints folder
print("\n=== ALL CHECKPOINTS ===")
ckpt_dir = Path('/content/drive/MyDrive/data/checkpoints')
for f in sorted(ckpt_dir.glob('*.pth')):
    try:
        c = torch.load(str(f), map_location='cpu')
        print(f"  {f.name} → epoch={c['epoch']}  val_acc={c['val_acc']:.4f}")
    except:
        print(f"  {f.name} → ❌ corrupted")

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
# ── Gradio Frontend for Deepfake Detection ─────────────────────────────────
!pip install gradio -q

import gradio as gr
import os, subprocess, shutil, tempfile
from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np
import torch
import cv2


def predict_video_gradio(video_path):
    """
    Wrapper around the existing pipeline.
    Input : video file path (provided by Gradio)
    Output: formatted prediction string
    """
    if video_path is None:
        return "❌ No video uploaded. Please upload a video file."

    video_path = str(video_path)

    if not os.path.exists(video_path):
        return f"❌ Video file not found: {video_path}"

    # ── Step 1: Extract frames (1 fps) ────────────────────────────────────
    tmp_frames = tempfile.mkdtemp(prefix="gradio_frames_")
    try:
        result = subprocess.run(
            [
                "ffmpeg", "-y", "-i", video_path,
                "-vf", "fps=1",
                "-q:v", "2",
                f"{tmp_frames}/frame_%04d.png",
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
        )
        frames = sorted(Path(tmp_frames).glob("frame_*.png"))

        if not frames:
            return (
                "❌ Could not extract any frames from the video.\n"
                "Please check that the file is a valid video (mp4, avi, mov, etc.)."
            )

        # ── Step 2: Detect faces & run model on each frame ────────────────
        MODEL.eval()
        frame_predictions = []
        frame_confidences = []
        no_face_count = 0

        with torch.no_grad():
            for frame_path in frames:
                # Detect + align face using the existing function
                face = detect_and_align(str(frame_path), MTCNN_DETECTOR, CFG["FACE_SIZE"])
                if face is None:
                    no_face_count += 1
                    continue

                # Apply SWT using the existing function
                swt_img = apply_swt(face, wavelet=CFG["WAVELET"], level=CFG["SWT_LEVEL"])
                pil = Image.fromarray((swt_img * 255).astype(np.uint8))
                tensor = EVAL_TRANSFORMS(pil).unsqueeze(0).to(DEVICE)

                # Model inference
                logits = MODEL(tensor)
                probs = torch.softmax(logits, dim=1).squeeze(0)
                pred_class = int(probs.argmax().item())
                confidence = float(probs[pred_class].item())

                frame_predictions.append(pred_class)
                frame_confidences.append(confidence)

        # ── Step 3: Aggregate predictions (majority vote) ─────────────────
        if not frame_predictions:
            return (
                f"⚠️  No faces detected in any of the {len(frames)} extracted frames.\n"
                "The video may not contain a clearly visible face."
            )

        vote = Counter(frame_predictions).most_common(1)
        final_pred = vote[0][0]
        final_name = CLASS_NAMES.get(final_pred, "UNKNOWN")
        avg_conf   = np.mean([
            c for p, c in zip(frame_predictions, frame_confidences)
            if p == final_pred
        ])

        # Per-frame breakdown
        per_frame = [CLASS_NAMES.get(p, "?") for p in frame_predictions]
        class_counts = Counter(per_frame)

        verdict_emoji = "✅ REAL" if final_pred == 0 else "🚨 MANIPULATED"

        output_lines = [
            f"{'='*50}",
            f"  DEEPFAKE DETECTION RESULT",
            f"{'='*50}",
            f"  Verdict        : {verdict_emoji}",
            f"  Prediction     : {final_name}",
            f"  Confidence     : {avg_conf:.1%}",
            f"{'─'*50}",
            f"  Frames analysed: {len(frame_predictions)} / {len(frames)}",
            f"  No face        : {no_face_count} frame(s)",
            f"  Frame votes    :",
        ]
        for cls_name, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
            bar = "█" * int(cnt / len(frame_predictions) * 20)
            output_lines.append(f"    {cls_name:<12} {cnt:>3}x  {bar}")
        output_lines.append(f"{'='*50}")

        return "\n".join(output_lines)

    except Exception as e:
        return f"❌ Prediction failed: {type(e).__name__}: {e}"

    finally:
        shutil.rmtree(tmp_frames, ignore_errors=True)


# ── Gradio Interface ────────────────────────────────────────────────────────
iface = gr.Interface(
    fn=predict_video_gradio,
    inputs=gr.Video(label="Upload Video"),
    outputs=gr.Textbox(label="Detection Result", lines=18),
    title="🎭 Deepfake Detection",
    description=(
        "Upload a video to detect whether it is **REAL**, **DEEPFAKE**, "
        "**FACESWAP**, or **FACE2FACE** manipulated.\n\n"
        "The pipeline extracts frames at 1 fps, detects and aligns faces with MTCNN, "
        "applies a Stationary Wavelet Transform (SWT), then classifies with a "
        "Vision Transformer (ViT) trained on FaceForensics++."
    ),
    allow_flagging="never",
)

iface.launch(share=True, debug=False)


In [ ]:
# ── Gradio Frontend for Deepfake Detection ─────────────────────────────────
!pip install gradio -q

import gradio as gr
import os, subprocess, shutil, tempfile
from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np
import torch
import cv2


def predict_video_gradio(video_path):
    """
    Wrapper around the existing pipeline.
    Input : video file path (provided by Gradio)
    Output: formatted prediction string
    """
    if video_path is None:
        return "❌ No video uploaded. Please upload a video file."

    video_path = str(video_path)

    if not os.path.exists(video_path):
        return f"❌ Video file not found: {video_path}"

    # ── Step 1: Extract frames (1 fps) ────────────────────────────────────
    tmp_frames = tempfile.mkdtemp(prefix="gradio_frames_")
    try:
        subprocess.run(
            [
                "ffmpeg", "-y", "-i", video_path,
                "-vf", "fps=1",
                "-q:v", "2",
                f"{tmp_frames}/frame_%04d.png",
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
        )
        frames = sorted(Path(tmp_frames).glob("frame_*.png"))

        if not frames:
            return (
                "❌ Could not extract any frames from the video.\n"
                "Please check that the file is a valid video (mp4, avi, mov, etc.)."
            )

        # ── Step 2: Detect faces & run model on each frame ────────────────
        MODEL.eval()
        frame_predictions = []
        frame_confidences = []
        no_face_count = 0

        with torch.no_grad():
            for frame_path in frames:
                face = detect_and_align(str(frame_path), MTCNN_DETECTOR, CFG["FACE_SIZE"])
                if face is None:
                    no_face_count += 1
                    continue

                swt_img = apply_swt(face, wavelet=CFG["WAVELET"], level=CFG["SWT_LEVEL"])
                pil = Image.fromarray((swt_img * 255).astype(np.uint8))
                tensor = EVAL_TRANSFORMS(pil).unsqueeze(0).to(DEVICE)

                logits = MODEL(tensor)
                probs = torch.softmax(logits, dim=1).squeeze(0)
                pred_class = int(probs.argmax().item())
                confidence = float(probs[pred_class].item())

                frame_predictions.append(pred_class)
                frame_confidences.append(confidence)

        # ── Step 3: Aggregate predictions (majority vote) ─────────────────
        if not frame_predictions:
            return (
                f"⚠️  No faces detected in any of the {len(frames)} extracted frames.\n"
                "The video may not contain a clearly visible face."
            )

        vote = Counter(frame_predictions).most_common(1)
        final_pred = vote[0][0]
        final_name = CLASS_NAMES.get(final_pred, "UNKNOWN")
        avg_conf = np.mean([
            c for p, c in zip(frame_predictions, frame_confidences)
            if p == final_pred
        ])

        per_frame = [CLASS_NAMES.get(p, "?") for p in frame_predictions]
        class_counts = Counter(per_frame)

        verdict_emoji = "✅ REAL" if final_pred == 0 else "🚨 MANIPULATED"

        output_lines = [
            f"{'='*50}",
            f"  DEEPFAKE DETECTION RESULT",
            f"{'='*50}",
            f"  Verdict        : {verdict_emoji}",
            f"  Prediction     : {final_name}",
            f"  Confidence     : {avg_conf:.1%}",
            f"{'─'*50}",
            f"  Frames analysed: {len(frame_predictions)} / {len(frames)}",
            f"  No face        : {no_face_count} frame(s)",
            f"  Frame votes    :",
        ]
        for cls_name, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
            bar = "█" * int(cnt / len(frame_predictions) * 20)
            output_lines.append(f"    {cls_name:<12} {cnt:>3}x  {bar}")
        output_lines.append(f"{'='*50}")

        return "\n".join(output_lines)

    except Exception as e:
        return f"❌ Prediction failed: {type(e).__name__}: {e}"

    finally:
        shutil.rmtree(tmp_frames, ignore_errors=True)


# ── Gradio Blocks UI (side-by-side layout) ─────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="Deepfake Detection") as iface:

    gr.Markdown(
        """
        # 🎭 Deepfake Detection
        Upload a video to detect whether it is **REAL**, **DEEPFAKE**, **FACESWAP**, or **FACE2FACE** manipulated.
        The pipeline extracts frames at 1 fps, detects faces with MTCNN, applies SWT, then classifies with a ViT trained on FaceForensics++.
        """
    )

    with gr.Row(equal_height=True):

        # ── Left column: video upload ──────────────────────────────────────
        with gr.Column(scale=1):
            video_input = gr.Video(label="📤 Upload Video")
            with gr.Row():
                clear_btn  = gr.Button("Clear",  variant="secondary")
                submit_btn = gr.Button("Submit", variant="primary")

        # ── Right column: result ───────────────────────────────────────────
        with gr.Column(scale=1):
            result_box = gr.Textbox(
                label="🔍 Detection Result",
                lines=20,
                max_lines=20,
                interactive=False,
                placeholder="Results will appear here after you submit a video...",
            )

    # ── Wire up buttons ───────────────────────────────────────────────────
    submit_btn.click(
        fn=predict_video_gradio,
        inputs=video_input,
        outputs=result_box,
    )
    clear_btn.click(
        fn=lambda: (None, ""),
        inputs=None,
        outputs=[video_input, result_box],
    )

iface.launch(share=True, debug=False)
